In [1]:
%load_ext autoreload
%autoreload 2

import matplotlib
matplotlib.use('Qt5Agg')

from eeg_toolkit import (
    load_config,
    find_subjects,
    compute_evokeds_subject,
    compute_evokeds_all,
    compute_grand_averages,
    load_grand_averages,
    plot_butterfly,
    plot_topomaps,
    plot_channel_overlay_with_ci,
    plot_roi_overlay,
)

cfg = load_config('../../../configs/eye_eeg_simul.yaml')
cfg_erp = load_config('../../../configs/erp_position.yaml')

subjects = find_subjects(cfg)
print(f"Subjects: {len(subjects)}")
print(f"Analysis: {getattr(cfg_erp.erp, 'analysis_name', '(default)')}")

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj17', 'subj2', 'subj20', 'subj28', 'subj7']
Subjects: 30
Analysis: position


In [2]:
test_subject = subjects[0]

ok, counts = compute_evokeds_subject(
    cfg, cfg_erp, test_subject,
    overwrite=True, verbose=True,
)

from eeg_toolkit.evoked import get_evoked_path
from eeg_toolkit.io import get_subject_dir

expected = get_evoked_path(cfg, test_subject, 'cue', cfg_erp)
print(f"\nExpected: {expected.name}")
print(f"Exists: {expected.exists()}")

print(f"\nFiles in {test_subject}:")
for f in sorted(get_subject_dir(cfg, test_subject).glob("*ave.fif")):
    print(f"   {f.name}")

   [subj3] cue: spatial_target_pos1=28, spatial_target_pos2=31, spatial_target_pos3=30, spatial_target_pos4=31, spatial_distractor_dig1=23, spatial_distractor_dig2=30, spatial_distractor_dig3=27, spatial_distractor_dig4=40, symbolic_target_dig1=33, symbolic_target_dig2=27, symbolic_target_dig3=30, symbolic_target_dig4=30, symbolic_distractor_pos1=31, symbolic_distractor_pos2=34, symbolic_distractor_pos3=29, symbolic_distractor_pos4=26
   [subj3] cue: saved 16 evokeds (16 conditions + 0 contrasts)

Expected: subj3_cue_position_ave.fif
Exists: True

Files in subj3:
   subj3_cue_ave.fif
   subj3_cue_position_ave.fif
   subj3_trial_ave.fif


In [3]:
evoked_summary = compute_evokeds_all(cfg, cfg_erp, overwrite=True, verbose=True)
compute_grand_averages(cfg, cfg_erp, overwrite=True, verbose=True)

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj17', 'subj2', 'subj20', 'subj28', 'subj7']
Computing evokeds for 30 subject(s)

--- [1/30] subj3 ---
   [subj3] cue: spatial_target_pos1=28, spatial_target_pos2=31, spatial_target_pos3=30, spatial_target_pos4=31, spatial_distractor_dig1=23, spatial_distractor_dig2=30, spatial_distractor_dig3=27, spatial_distractor_dig4=40, symbolic_target_dig1=33, symbolic_target_dig2=27, symbolic_target_dig3=30, symbolic_target_dig4=30, symbolic_distractor_pos1=31, symbolic_distractor_pos2=34, symbolic_distractor_pos3=29, symbolic_distractor_pos4=26
   [subj3] cue: saved 16 evokeds (16 conditions + 0 contrasts)

--- [2/30] subj4 ---
   [subj4] cue: spatial_target_pos1=28, spatial_target_pos2=31, spatial_target_pos3=30, spatial_target_pos4=31, spatial_distractor_dig1=30, spatial_distractor_dig2=28, spatial_distractor_dig3=33, spatial_distractor_dig4=29, symbolic_target_dig1=33, symbolic_target_dig2=27, symbolic_target_dig3=30, s

True

In [4]:
import numpy as np
import mne
from eeg_toolkit.io import find_subjects
from eeg_toolkit.evoked import get_evoked_path

# Define how to collapse position-level conditions into left/right groups
groupings = {
    'spatial_target_left':       ['spatial_target_pos1', 'spatial_target_pos2'],
    'spatial_target_right':      ['spatial_target_pos3', 'spatial_target_pos4'],
    'spatial_distractor_left':   ['spatial_distractor_dig1', 'spatial_distractor_dig2'],
    'spatial_distractor_right':  ['spatial_distractor_dig3', 'spatial_distractor_dig4'],
    'symbolic_target_left':      ['symbolic_target_dig1', 'symbolic_target_dig2'],
    'symbolic_target_right':     ['symbolic_target_dig3', 'symbolic_target_dig4'],
    'symbolic_distractor_left':  ['symbolic_distractor_pos1', 'symbolic_distractor_pos2'],
    'symbolic_distractor_right': ['symbolic_distractor_pos3', 'symbolic_distractor_pos4'],
}

subjects = find_subjects(cfg)
collapsed = {grp: [] for grp in groupings}

for subject in subjects:
    evo_path = get_evoked_path(cfg, subject, 'cue', cfg_erp)
    if not evo_path.exists():
        continue
    subj = {e.comment: e for e in mne.read_evokeds(evo_path, verbose='WARNING')}

    for grp_name, components in groupings.items():
        evos = [subj[c] for c in components if c in subj]
        if len(evos) == len(components):
            combined = mne.combine_evoked(evos, weights='equal')
            combined.comment = grp_name
            collapsed[grp_name].append(combined)

print(f"Subjects per group:")
for grp, lst in collapsed.items():
    print(f"   {grp}: N={len(lst)}")

[find_subjects] excluded 8 subject(s): ['subj13', 'subj15', 'subj16', 'subj17', 'subj2', 'subj20', 'subj28', 'subj7']
Subjects per group:
   spatial_target_left: N=30
   spatial_target_right: N=30
   spatial_distractor_left: N=30
   spatial_distractor_right: N=30
   symbolic_target_left: N=30
   symbolic_target_right: N=30
   symbolic_distractor_left: N=30
   symbolic_distractor_right: N=30


In [5]:
import matplotlib.pyplot as plt

# ROIs to compare laterality at
rois = {
    'posterior_left':  ['P3', 'P7', 'O1'],
    'posterior_right': ['P4', 'P8', 'O2'],
}

# Plot pairs: each entry is (title, [left_group, right_group])
plot_groups = [
    ('Spatial — Target',      ['spatial_target_left',     'spatial_target_right']),
    ('Spatial — Distractor',  ['spatial_distractor_left', 'spatial_distractor_right']),
    ('Symbolic — Target',     ['symbolic_target_left',    'symbolic_target_right']),
    ('Symbolic — Distractor', ['symbolic_distractor_left','symbolic_distractor_right']),
]

for title, groups in plot_groups:
    evokeds_to_plot = {grp: collapsed[grp] for grp in groups}
    for roi_name, channels in rois.items():
        valid_ch = [c for c in channels
                    if c in collapsed[groups[0]][0].ch_names]
        fig = mne.viz.plot_compare_evokeds(
            evokeds_to_plot,
            picks=valid_ch,
            combine='mean',
            title=f"{title} — {roi_name}",
            ci=0.95,
            show=False,
        )
        plt.show()

combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
combining channels using "mean"
